In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("Cross_System_Monitoring")
    # Delta Lake Configurations
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.file.impl", "org.apache.hadoop.fs.local.LocalFs")
    # Network configurations to prevent Py4J Java timeouts
    .config("spark.network.timeout", "600s")
    .config("spark.executor.heartbeatInterval", "60s")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

load delta tables

In [2]:
billing = spark.read.format("delta").load("../silver/billing")

analytics = spark.read.format("delta").load("../silver/analytics")

calculating revenue

In [3]:
from pyspark.sql.functions import sum

daily_revenue = billing.groupBy(
    "transaction_date"
).agg(
    sum("amount").alias("billing_revenue")
)

comparsion

In [4]:
comparison = daily_revenue.join(
    analytics,
    daily_revenue.transaction_date == analytics.date
)

drift

In [5]:
from pyspark.sql.functions import abs

drift = comparison.withColumn(
    "difference",
    abs(
        comparison.billing_revenue -
        comparison.total_revenue
    )
)

drift.show()

+----------------+------------------+----------+---------------+-------------+---------------+------------------+
|transaction_date|   billing_revenue|      date|total_customers|total_revenue|avg_transaction|        difference|
+----------------+------------------+----------+---------------+-------------+---------------+------------------+
|      2023-07-15|           4946.72|2023-07-15|              9|      3701.48|         371.13|1245.2400000000002|
|      2022-03-28|3227.5899999999997|2022-03-28|              8|      1755.72|         271.74|1471.8699999999997|
|      2023-06-22| 6440.889999999999|2023-06-22|             13|      2781.67|          249.9|3659.2199999999993|
|      2022-07-31| 7255.839999999999|2022-07-31|             12|      6700.04|         871.24| 555.7999999999993|
|      2022-11-29|          11285.22|2022-11-29|             16|      9770.01|         743.25|1515.2099999999991|
|      2023-09-14|4198.4800000000005|2023-09-14|             12|      3762.74|         5

saving drift reports

In [6]:
drift.write.format("delta").mode("overwrite").save("../gold/drift_report")